<a href="https://colab.research.google.com/github/ekonjmrivas-devops/llm_engineering/blob/mis-ejercicios/W3_PRACTICA_DATOS_SINTETICOS_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semana 3 - Práctica | Generador de Datos Sintéticos (v2)

**Bloque - Librerías**

In [1]:
!pip install -q anthropic


In [2]:
!pip install -q -U "bitsandbytes>=0.46.1"


In [3]:
!pip install -q gradio


In [4]:
import os
import re
import json
import csv
import time
import threading
from datetime import datetime
from google.colab import drive
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import anthropic
from pydantic import ValidationError, create_model
import gradio as gr


**Bloque - Setup y conexión a Drive**

In [5]:
# Conexión a Google Drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Rutas fijas del proyecto
BASE_PATH = '/content/drive/MyDrive/cursollms/week3_sintetico'
SEED_PATH = f'{BASE_PATH}/Seed files'
SCHEMA_PATH = f'{BASE_PATH}/Schema files'
PROMPT_PATH = f'{BASE_PATH}/Prompt files'
OUTPUT_PATH = f'{BASE_PATH}/Outbound files'


In [7]:
# Crear carpetas si no existen
for carpeta in (SEED_PATH, SCHEMA_PATH, PROMPT_PATH, OUTPUT_PATH):
    os.makedirs(carpeta, exist_ok=True)


In [8]:
print(f"Semillas: {SEED_PATH}")
print(f"Esquemas: {SCHEMA_PATH}")
print(f"Prompts:  {PROMPT_PATH}")
print(f"Salida:   {OUTPUT_PATH}")


Semillas: /content/drive/MyDrive/cursollms/week3_sintetico/Seed files
Esquemas: /content/drive/MyDrive/cursollms/week3_sintetico/Schema files
Prompts:  /content/drive/MyDrive/cursollms/week3_sintetico/Prompt files
Salida:   /content/drive/MyDrive/cursollms/week3_sintetico/Outbound files


**Bloque - Listar archivos de semilla disponibles en Drive**

In [9]:
def listar_archivos_semilla():
    """Lista los archivos JSON disponibles en Seed files, para el Dropdown de Gradio."""
    archivos = [f for f in os.listdir(SEED_PATH) if f.lower().endswith(".json")]
    archivos.sort()
    return archivos


**Bloque - Carga y guardado del dataset semilla en Drive**

In [10]:
def cargar_dataset_semilla(nombre_archivo):
    """
    Carga el dataset semilla desde un archivo JSON depositado en Seed files.
    nombre_archivo: nombre del archivo dentro de SEED_PATH.
    """
    ruta = os.path.join(SEED_PATH, nombre_archivo)
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró '{nombre_archivo}' en {SEED_PATH}")
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)


In [11]:
def guardar_dataset_semilla(datos, nombre_archivo):
    """
    Guarda una lista de registros semilla como JSON en Seed files.
    nombre_archivo: nombre de archivo (con o sin extensión .json).
    """
    if not nombre_archivo.lower().endswith(".json"):
        nombre_archivo += ".json"
    ruta = os.path.join(SEED_PATH, nombre_archivo)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)
    print(f"✅ Semilla guardada en: {ruta}")
    return ruta


**Bloque - Listar, cargar y guardar esquemas de validación en Drive**

In [12]:
def listar_archivos_esquema():
    """Lista los archivos JSON disponibles en Schema files."""
    archivos = [f for f in os.listdir(SCHEMA_PATH) if f.lower().endswith(".json")]
    archivos.sort()
    return archivos


In [13]:
def cargar_esquema(nombre_archivo):
    """
    Carga un esquema de validación desde Schema files.
    Formato esperado: {"campos": [{"nombre":..., "tipo":...}, ...], "campo_id": "nombre_o_null"}
    """
    ruta = os.path.join(SCHEMA_PATH, nombre_archivo)
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró '{nombre_archivo}' en {SCHEMA_PATH}")
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)


In [14]:
def guardar_esquema(esquema, nombre_archivo):
    """Guarda un esquema de validación como JSON en Schema files."""
    if not nombre_archivo.lower().endswith(".json"):
        nombre_archivo += ".json"
    ruta = os.path.join(SCHEMA_PATH, nombre_archivo)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(esquema, f, ensure_ascii=False, indent=2)
    print(f"✅ Esquema guardado en: {ruta}")
    return ruta


**Bloque - Construcción dinámica del modelo de validación (Pydantic)**

In [15]:
TIPO_MAP = {"str": str, "int": int, "float": float, "bool": bool}

def construir_modelo_validacion(esquema):
    """
    Construye una clase Pydantic en tiempo de ejecución a partir de un esquema
    genérico (lista de campos con nombre y tipo). Sustituye a una clase fija
    como la anterior "TicketSintetico", permitiendo validar cualquier dominio.
    """
    campos_modelo = {}
    for campo in esquema["campos"]:
        tipo_python = TIPO_MAP.get(campo["tipo"], str)
        campos_modelo[campo["nombre"]] = (tipo_python, ...)
    return create_model("ModeloDinamico", **campos_modelo)


**Bloque - Listar, cargar y guardar prompts de sistema en Drive**

In [16]:
def listar_archivos_prompt():
    """Lista los archivos JSON disponibles en Prompt files."""
    archivos = [f for f in os.listdir(PROMPT_PATH) if f.lower().endswith(".json")]
    archivos.sort()
    return archivos


In [17]:
def cargar_prompt(nombre_archivo):
    """Carga un prompt de sistema guardado (formato {"contenido": texto})."""
    ruta = os.path.join(PROMPT_PATH, nombre_archivo)
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró '{nombre_archivo}' en {PROMPT_PATH}")
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)["contenido"]


In [18]:
def guardar_prompt(texto, nombre_archivo):
    """Guarda un prompt de sistema como JSON en Prompt files."""
    if not nombre_archivo.lower().endswith(".json"):
        nombre_archivo += ".json"
    ruta = os.path.join(PROMPT_PATH, nombre_archivo)
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump({"contenido": texto}, f, ensure_ascii=False, indent=2)
    print(f"✅ Prompt guardado en: {ruta}")
    return ruta


**Bloque - Prompt de sistema automático a partir del esquema**

In [19]:
def construir_prompt_sistema_automatico(esquema):
    """
    Genera un system prompt genérico a partir de los campos del esquema.
    Se usa como valor por defecto cuando el usuario no ha seleccionado
    ni escrito un prompt propio en la pestaña correspondiente.
    """
    campos_txt = ", ".join(c["nombre"] for c in esquema["campos"])

    prompt = f"""Eres un generador de datos sintéticos. Tu tarea es crear registros NUEVOS
y REALISTAS basados en el estilo, formato y variedad de los ejemplos semilla proporcionados.

Reglas estrictas:
1. Responde ÚNICAMENTE con un array JSON válido, sin texto adicional ni explicaciones.
2. Cada registro debe tener exactamente estos campos: {campos_txt}.
3. NO repitas literalmente los ejemplos; genera situaciones distintas pero plausibles dentro del mismo dominio.
4. Mantén coherencia de estilo, longitud y nivel de detalle con los ejemplos semilla."""

    campo_id = esquema.get("campo_id")
    if campo_id:
        prompt += f"\n5. El campo '{campo_id}' debe continuar la numeración/formato del último ejemplo semilla."

    return prompt


**Bloque - Cálculo genérico del siguiente ID**

In [20]:
def calcular_siguiente_id(seed_dataset, campo_id):
    """
    Calcula el siguiente valor de ID a partir del último ejemplo semilla,
    detectando el número final de la cadena (ej. 'TCK-1001' -> 'TCK-1002').
    Devuelve None si el esquema no define campo_id o no se detecta un patrón numérico.
    """
    if not campo_id or not seed_dataset:
        return None

    ultimo_valor = str(seed_dataset[-1].get(campo_id, ""))
    match = re.search(r"(\d+)$", ultimo_valor)
    if not match:
        return None

    numero = match.group(1)
    prefijo = ultimo_valor[:match.start()]
    siguiente_numero = str(int(numero) + 1).zfill(len(numero))
    return f"{prefijo}{siguiente_numero}"


**Bloque - Construcción del prompt de usuario**

In [21]:
def construir_prompt_usuario(seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales=""):
    """
    Construye el prompt de usuario con los ejemplos semilla (few-shot),
    la petición de generación y, opcionalmente, instrucciones adicionales
    escritas por el usuario en la pestaña principal.
    """
    prompt = f"""Aquí tienes ejemplos semilla de referencia:

{json.dumps(seed_dataset, indent=2, ensure_ascii=False)}

Genera {num_ejemplos} registros NUEVOS siguiendo el mismo estilo y estructura."""

    if siguiente_id:
        prompt += f"\nEmpieza la numeración en: {siguiente_id}"

    if instrucciones_adicionales and instrucciones_adicionales.strip():
        prompt += f"\n\nInstrucciones adicionales:\n{instrucciones_adicionales.strip()}"

    prompt += "\n\nResponde solo con el array JSON."
    return prompt


**Bloque - Configuración de cuantización y carga de Llama local**

In [22]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)


In [23]:
# Carga el modelo de Llama solo cuando sea necesario (idempotente)

LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

llama_tokenizer = None
llama_model = None

def cargar_llama_si_necesario():
    """
    Carga el tokenizer y el modelo Llama en memoria si aún no están cargados.
    Si ya se cargaron en esta sesión, la llamada es prácticamente instantánea.
    """
    global llama_tokenizer, llama_model
    if llama_model is None:
        llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA)
        llama_model = AutoModelForCausalLM.from_pretrained(
            LLAMA,
            device_map="auto",
            quantization_config=quant_config
        )


**Bloque - Configuración Claude**

In [24]:
CLAUDE = "claude-sonnet-4-6"

claude_client = anthropic.Anthropic(
    api_key=userdata.get('ANTHROPIC_API_KEY')
)


**Bloque - Generación con Claude**

In [25]:
def generar_con_claude(system_prompt, seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales=""):
    """
    Genera registros sintéticos usando Claude API.
    Devuelve el texto crudo de la respuesta (a validar después).
    """
    user_prompt = construir_prompt_usuario(seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales)

    response = claude_client.messages.create(
        model=CLAUDE,
        max_tokens=4000,
        system=system_prompt,
        messages=[{"role": "user", "content": user_prompt}]
    )

    return response.content[0].text


**Bloque - Generación con Llama**

In [26]:
def generar_con_llama(system_prompt, seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales=""):
    """
    Genera registros sintéticos usando Llama local.
    Devuelve el texto crudo de la respuesta (a validar después).
    """
    cargar_llama_si_necesario()

    user_prompt = construir_prompt_usuario(seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales)

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system_prompt}<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_prompt}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""

    inputs = llama_tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=4000
    )

    return llama_tokenizer.decode(outputs[0], skip_special_tokens=True)


**Bloque - Función orquestadora del modelo**

In [27]:
def generar_registros_raw(system_prompt, seed_dataset, num_ejemplos, siguiente_id, modelo="claude", instrucciones_adicionales=""):
    """
    Genera registros con el modelo seleccionado.
    modelo: "claude" o "llama"
    """
    if modelo == "claude":
        return generar_con_claude(system_prompt, seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales)
    elif modelo == "llama":
        return generar_con_llama(system_prompt, seed_dataset, num_ejemplos, siguiente_id, instrucciones_adicionales)
    else:
        raise ValueError(f"Modelo no reconocido: {modelo}")


**Bloque - Parseo y validación de la salida**

In [28]:
def parsear_y_validar(texto_generado, modelo_validacion):
    """
    Limpia la salida del modelo, la parsea como JSON y valida cada registro
    contra el modelo Pydantic dinámico construido a partir del esquema activo.
    Devuelve (registros_validos, errores).
    """
    limpio = texto_generado.replace("```json", "").replace("```", "").strip()

    try:
        datos = json.loads(limpio)
    except json.JSONDecodeError as e:
        return [], [f"Error de parseo JSON: {e}"]

    registros_validos = []
    errores = []

    for item in datos:
        try:
            registro = modelo_validacion(**item)
            registros_validos.append(registro.model_dump())
        except ValidationError as e:
            errores.append(f"Registro inválido ({item}): {e}")

    return registros_validos, errores


**Bloque - Generación por lotes**

In [29]:
TAMANO_LOTE = 20  # nº máximo de registros por llamada al modelo

def generar_dataset_por_lotes(seed_dataset, esquema, system_prompt, num_total, modelo="claude", instrucciones_adicionales=""):
    """
    Genera el número total de registros solicitado, dividiendo la petición en lotes.
    Usa el esquema activo para construir el validador dinámico y calcular
    la continuación del campo ID (si el esquema lo define).
    Devuelve (registros_generados, errores_acumulados).
    """
    modelo_validacion = construir_modelo_validacion(esquema)
    campo_id = esquema.get("campo_id")

    contexto_seed = list(seed_dataset)
    registros_generados = []
    errores_acumulados = []
    pendientes = num_total

    while pendientes > 0:
        lote = min(TAMANO_LOTE, pendientes)
        siguiente_id = calcular_siguiente_id(contexto_seed, campo_id)

        print(f"⏳ Generando lote de {lote} registros...")
        texto_crudo = generar_registros_raw(
            system_prompt, contexto_seed, lote, siguiente_id,
            modelo=modelo, instrucciones_adicionales=instrucciones_adicionales
        )
        validos, errores = parsear_y_validar(texto_crudo, modelo_validacion)

        registros_generados.extend(validos)
        errores_acumulados.extend(errores)
        if validos and campo_id:
            contexto_seed = contexto_seed + validos  # continúa la numeración en el siguiente lote
        pendientes -= lote

        print(f"✅ Lote completado: {len(validos)} válidos, {len(errores)} con error")

    return registros_generados, errores_acumulados


**Bloque - Funciones auxiliares con IA (enriquecer prompt, validar esquema, formatear semilla)**

In [30]:
def enriquecer_prompt_con_ia(texto_borrador):
    """
    Envía un borrador de system prompt a Claude para enriquecerlo:
    más claro, más completo, sin perder la intención original.
    """
    instruccion = f"""Eres un experto en prompt engineering. Mejora el siguiente system prompt
para un generador de datos sintéticos: hazlo más claro, completo y con reglas explícitas de formato,
sin cambiar su intención original. Responde ÚNICAMENTE con el prompt mejorado, sin explicaciones.

Prompt original:
{texto_borrador}
"""
    response = claude_client.messages.create(
        model=CLAUDE,
        max_tokens=1500,
        messages=[{"role": "user", "content": instruccion}]
    )
    return response.content[0].text.strip()


In [31]:
def validar_formatear_esquema_con_ia(texto_borrador):
    """
    Envía una descripción libre de campos a Claude y devuelve un esquema
    JSON válido con el formato {"campos": [{"nombre":..., "tipo":...}], "campo_id": ...}.
    """
    instruccion = f"""Convierte la siguiente descripción de campos en un esquema JSON válido
con este formato exacto:
{{"campos": [{{"nombre": "...", "tipo": "str|int|float|bool"}}, ...], "campo_id": "nombre_del_campo_id_o_null"}}

Responde ÚNICAMENTE con el JSON, sin texto adicional.

Descripción:
{texto_borrador}
"""
    response = claude_client.messages.create(
        model=CLAUDE,
        max_tokens=1000,
        messages=[{"role": "user", "content": instruccion}]
    )
    limpio = response.content[0].text.replace("```json", "").replace("```", "").strip()
    return json.loads(limpio)  # lanza excepción si la IA no devolvió JSON válido


In [32]:
def formatear_semilla_con_ia(texto_borrador, esquema=None):
    """
    Envía datos semilla pegados en bruto (texto libre, CSV, etc.) a Claude
    y devuelve una lista de registros en JSON, ajustada al esquema si se indica.
    """
    contexto_esquema = ""
    if esquema:
        campos_txt = ", ".join(c["nombre"] for c in esquema["campos"])
        contexto_esquema = f"\nLos registros deben tener exactamente estos campos: {campos_txt}."

    instruccion = f"""Convierte los siguientes datos en una lista JSON de registros bien formada.
{contexto_esquema}
Responde ÚNICAMENTE con el array JSON, sin texto adicional.

Datos:
{texto_borrador}
"""
    response = claude_client.messages.create(
        model=CLAUDE,
        max_tokens=2000,
        messages=[{"role": "user", "content": instruccion}]
    )
    limpio = response.content[0].text.replace("```json", "").replace("```", "").strip()
    return json.loads(limpio)


**Bloque - Generación automática del nombre del archivo**

In [33]:
def generar_nombre_archivo(prefijo="dataset_sintetico", extension="json"):
    """
    Genera un nombre de archivo de salida basado en un prefijo y timestamp.
    Ejemplo: dataset_sintetico_20260703_143052.json
    """
    ahora = datetime.now()
    timestamp = ahora.strftime("%Y%m%d_%H%M%S")
    return f"{prefijo}_{timestamp}.{extension}"


**Bloque - Guardar resultado en Google Drive**

In [34]:
def guardar_json(registros, nombre_archivo):
    """Guarda la lista de registros en un archivo JSON en la carpeta de salida."""
    ruta_completa = os.path.join(OUTPUT_PATH, nombre_archivo)

    with open(ruta_completa, "w", encoding="utf-8") as f:
        json.dump(registros, f, ensure_ascii=False, indent=2)

    print(f"✅ Archivo JSON guardado en: {ruta_completa}")
    return ruta_completa


In [35]:
def guardar_csv(registros, nombre_archivo):
    """Guarda la lista de registros en un archivo CSV en la carpeta de salida."""
    ruta_completa = os.path.join(OUTPUT_PATH, nombre_archivo)
    campos = list(registros[0].keys()) if registros else []

    with open(ruta_completa, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        writer.writeheader()
        writer.writerows(registros)

    print(f"✅ Archivo CSV guardado en: {ruta_completa}")
    return ruta_completa


**Bloque - Función orquestadora para guardar archivo**

In [36]:
def guardar_resultado(registros, nombre_archivo, formato="json"):
    """
    Guarda el resultado en el formato indicado.
    formato: "json" o "csv"
    """
    if formato == "json":
        return guardar_json(registros, nombre_archivo)
    elif formato == "csv":
        return guardar_csv(registros, nombre_archivo)
    else:
        raise ValueError(f"Formato no reconocido: {formato}")


**Bloque - Función orquestadora principal**

In [37]:
def generar_dataset_sintetico(num_ejemplos, modelo, formato_salida, nombre_semilla, esquema, system_prompt,
                                nombre_salida=None, instrucciones_adicionales=""):
    """
    Orquesta el flujo completo de generación de un dataset sintético.

    Parámetros obligatorios (deben venir ya resueltos desde las pestañas de gestión):
    - nombre_semilla: fichero de Seed files ya seleccionado/creado
    - esquema: dict del esquema ya cargado (desde Schema files o recién creado)
    - system_prompt: texto de prompt de sistema ya resuelto (auto-generado o editado)

    Devuelve: (registros, ruta_archivo_guardado, errores)
    """
    if not nombre_semilla or not esquema or not system_prompt:
        raise ValueError("Faltan parámetros obligatorios: semilla, esquema y prompt de sistema.")

    print(f"🔧 Ejemplos solicitados: {num_ejemplos} | Modelo: {modelo} | Formato: {formato_salida}")
    print("─" * 50)

    # Paso 1 — Cargar dataset semilla
    print("⏳ Paso 1: Cargando dataset semilla...")
    seed_dataset = cargar_dataset_semilla(nombre_semilla)
    print(f"✅ Semilla cargada ({len(seed_dataset)} ejemplos)")

    # Paso 2 — Generar registros por lotes con validación dinámica
    print(f"⏳ Paso 2: Generando registros con {modelo}...")
    registros, errores = generar_dataset_por_lotes(
        seed_dataset, esquema, system_prompt, num_ejemplos,
        modelo=modelo, instrucciones_adicionales=instrucciones_adicionales
    )
    print(f"✅ Generación completada: {len(registros)} registros válidos, {len(errores)} errores")

    # Paso 3 — Generar nombre de archivo si no se proporcionó
    if nombre_salida is None or nombre_salida.strip() == "":
        nombre_salida = generar_nombre_archivo(extension=formato_salida)
    else:
        nombre_salida = f"{os.path.splitext(nombre_salida)[0]}.{formato_salida}"

    print(f"⏳ Paso 3: Guardando resultado como {nombre_salida}...")
    ruta = guardar_resultado(registros, nombre_salida, formato=formato_salida)
    print(f"✅ Archivo guardado en: {ruta}")
    print("─" * 50)
    print("🎉 Proceso completado")

    return registros, ruta, errores


**Bloque - Función de arranque del sistema (pantalla de carga)**

In [38]:
import subprocess

def obtener_estado_recursos():
    """
    Devuelve un resumen compacto del estado de la GPU (memoria usada/total, % uso),
    equivalente a `!nvidia-smi` pero capturable como texto dentro de una función.
    """
    try:
        resultado = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
             "--format=csv,noheader"],
            capture_output=True, text=True, check=True
        )
        return resultado.stdout.strip()
    except Exception as e:
        return f"No se pudo obtener el estado de la GPU: {e}"

In [39]:
def cargar_sistema():
    """
    Función generadora que ejecuta la carga inicial del sistema y va
    emitiendo mensajes de estado para la pantalla de carga de Gradio.

    Carga Llama de forma eager (al arrancar) porque su tiempo (~12 min) es
    el cuello de botella que se quiere visualizar en el log. Si en tu caso
    de uso solo necesitas Claude, comenta la llamada a cargar_llama_si_necesario()
    para arrancar en segundos.
    """

    log = "⏳ Comprobando conexión a Drive...\n"
    yield log

    log += "✅ Drive conectado.\n⏳ Configurando cliente Claude...\n"
    yield log

    log += "✅ Cliente Claude listo.\n⏳ Cargando modelo Llama local (puede tardar ~12 min)...\n"
    yield log

    log += f"📊 GPU antes de cargar Llama:\n{obtener_estado_recursos()}\n"
    yield log

    resultado = {}
    def _cargar():
        #cargar_llama_si_necesario()
        resultado["listo"] = True

    hilo = threading.Thread(target=_cargar)
    hilo.start()

    inicio = time.time()
    while hilo.is_alive():
        time.sleep(5)
        transcurrido = int(time.time() - inicio)
        yield log + f"   ...cargando ({transcurrido // 60} min {transcurrido % 60} s transcurridos)"

    log += f"✅ Modelo Llama cargado ({int(time.time() - inicio)} s).\n🎉 Sistema listo.\n"
    yield log

    log += f"📊 GPU tras cargar Llama:\n{obtener_estado_recursos()}\n"
    yield log

**Bloque - Interfaz Gradio (pantalla de carga + pestañas de gestión)**

In [40]:
with gr.Blocks(title="Generador de Datos Sintéticos") as app:

    # Estado con los ficheros/valores activos para la generación
    semilla_activa = gr.State(None)     # nombre de archivo en Seed files
    esquema_activo = gr.State(None)     # dict del esquema cargado
    prompt_activo = gr.State(None)      # texto del system prompt resuelto

    # ── Pantalla de carga ──────────────────────────────────────
    with gr.Group(visible=True) as grupo_carga:
        gr.Markdown("# 🧪 Generador de Datos Sintéticos")
        gr.Markdown("Inicializando sistema, por favor espera...")
        log_carga = gr.Textbox(label="Registro de arranque", lines=10, interactive=False)

    # ── Formulario principal ───────────────────────────────────
    with gr.Group(visible=False) as grupo_principal:
        with gr.Tabs():

            # --- Pestaña: Prompt de sistema ---
            with gr.TabItem("📝 Prompt de sistema"):
                gr.Markdown("Selecciona un prompt existente o escribe uno nuevo.")
                prompt_dropdown = gr.Dropdown(choices=listar_archivos_prompt(), label="Prompts guardados")
                btn_refrescar_prompt = gr.Button("🔄 Refrescar", size="sm")
                prompt_texto = gr.Textbox(label="Contenido del prompt", lines=10)
                btn_cargar_prompt = gr.Button("📂 Cargar seleccionado")
                btn_mejorar_prompt = gr.Button("✨ Mejorar con IA")
                prompt_nombre_guardar = gr.Textbox(label="Nombre para guardar (opcional)")
                btn_guardar_prompt = gr.Button("💾 Guardar y usar", variant="primary")
                prompt_estado = gr.Textbox(label="Estado", interactive=False)

            # --- Pestaña: Esquema de validación ---
            with gr.TabItem("🧩 Esquema de validación"):
                gr.Markdown("Selecciona un esquema existente o describe los campos y valida con IA.")
                esquema_dropdown = gr.Dropdown(choices=listar_archivos_esquema(), label="Esquemas guardados")
                btn_refrescar_esquema = gr.Button("🔄 Refrescar", size="sm")
                esquema_texto = gr.Textbox(
                    label="Descripción de campos (o JSON ya formado)",
                    lines=8,
                    placeholder="Ej: id (texto), categoria (texto), prioridad (texto), importe (numero)..."
                )
                btn_cargar_esquema = gr.Button("📂 Cargar seleccionado")
                btn_validar_esquema = gr.Button("✨ Validar/formatear con IA")
                esquema_nombre_guardar = gr.Textbox(label="Nombre para guardar (opcional)")
                btn_guardar_esquema = gr.Button("💾 Guardar y usar", variant="primary")
                esquema_estado = gr.Textbox(label="Estado", interactive=False)

            # --- Pestaña: Dataset semilla ---
            with gr.TabItem("🌱 Dataset semilla"):
                gr.Markdown("Selecciona una semilla existente o pega datos en bruto y da formato con IA.")
                semilla_dropdown = gr.Dropdown(choices=listar_archivos_semilla(), label="Semillas guardadas")
                btn_refrescar_semilla = gr.Button("🔄 Refrescar", size="sm")
                semilla_texto = gr.Textbox(label="Datos semilla (pegar en bruto o JSON)", lines=10)
                btn_cargar_semilla = gr.Button("📂 Cargar seleccionada")
                btn_formatear_semilla = gr.Button("✨ Formatear con IA")
                semilla_nombre_guardar = gr.Textbox(label="Nombre para guardar (opcional)")
                btn_guardar_semilla = gr.Button("💾 Guardar y usar", variant="primary")
                semilla_estado = gr.Textbox(label="Estado", interactive=False)

            # --- Pestaña: Generar dataset ---
            with gr.TabItem("🚀 Generar dataset"):
                gr.Markdown("### Ficheros activos")
                with gr.Row():
                    semilla_activa_display = gr.Textbox(label="Semilla", interactive=False)
                    esquema_activo_display = gr.Textbox(label="Esquema", interactive=False)
                    prompt_activo_display = gr.Textbox(label="Prompt de sistema", interactive=False)

                gr.Markdown("### Configuración")
                num_ejemplos = gr.Number(value=10, label="Nº de ejemplos a generar", precision=0)
                modelo = gr.Radio(choices=["claude", "llama"], value="claude", label="Modelo de IA")
                formato_salida = gr.Radio(choices=["json", "csv"], value="json", label="Formato de salida")
                instrucciones_adicionales = gr.Textbox(
                    label="Instrucciones adicionales (opcional)",
                    placeholder="Ej: prioriza casos de categoría Finanzas"
                )
                nombre_salida = gr.Textbox(label="Nombre del archivo de salida (opcional)")

                btn_generar = gr.Button("🚀 Generar dataset", variant="primary")

                gr.Markdown("### Resultado")
                preview_output = gr.Textbox(label="Preview (primeros 5 registros)", lines=15)
                ruta_output = gr.Textbox(label="Archivo guardado en", interactive=False)
                estado_output = gr.Textbox(label="Estado", interactive=False)

    # ══════════════ Eventos: arranque ══════════════
    app.load(
        fn=cargar_sistema,
        outputs=log_carga
    ).then(
        fn=lambda: (gr.update(visible=False), gr.update(visible=True)),
        outputs=[grupo_carga, grupo_principal]
    )

    # ══════════════ Eventos: pestaña Prompt ══════════════
    btn_refrescar_prompt.click(fn=lambda: gr.update(choices=listar_archivos_prompt()), outputs=prompt_dropdown)
    btn_cargar_prompt.click(fn=lambda nombre: cargar_prompt(nombre) if nombre else "", inputs=prompt_dropdown, outputs=prompt_texto)
    btn_mejorar_prompt.click(fn=enriquecer_prompt_con_ia, inputs=prompt_texto, outputs=prompt_texto)

    def _guardar_prompt_y_usar(texto, nombre):
        nombre_final = nombre.strip() if nombre and nombre.strip() else generar_nombre_archivo("prompt", "json")
        guardar_prompt(texto, nombre_final)
        return texto, f"✅ Prompt activo: {nombre_final}", nombre_final

    btn_guardar_prompt.click(
        fn=_guardar_prompt_y_usar,
        inputs=[prompt_texto, prompt_nombre_guardar],
        outputs=[prompt_activo, prompt_estado, prompt_activo_display]
    )

    # ══════════════ Eventos: pestaña Esquema ══════════════
    btn_refrescar_esquema.click(fn=lambda: gr.update(choices=listar_archivos_esquema()), outputs=esquema_dropdown)

    def _cargar_esquema_ui(nombre):
        if not nombre:
            return ""
        esquema = cargar_esquema(nombre)
        return json.dumps(esquema, indent=2, ensure_ascii=False)

    btn_cargar_esquema.click(fn=_cargar_esquema_ui, inputs=esquema_dropdown, outputs=esquema_texto)

    def _validar_esquema_ui(texto):
        esquema = validar_formatear_esquema_con_ia(texto)
        return json.dumps(esquema, indent=2, ensure_ascii=False)

    btn_validar_esquema.click(fn=_validar_esquema_ui, inputs=esquema_texto, outputs=esquema_texto)

    def _guardar_esquema_y_usar(texto, nombre):
        esquema = json.loads(texto)  # debe ser ya JSON válido (tras validar con IA o pegado directo)
        nombre_final = nombre.strip() if nombre and nombre.strip() else generar_nombre_archivo("esquema", "json")
        guardar_esquema(esquema, nombre_final)
        return esquema, f"✅ Esquema activo: {nombre_final}", nombre_final

    btn_guardar_esquema.click(
        fn=_guardar_esquema_y_usar,
        inputs=[esquema_texto, esquema_nombre_guardar],
        outputs=[esquema_activo, esquema_estado, esquema_activo_display]
    )

    # ══════════════ Eventos: pestaña Semilla ══════════════
    btn_refrescar_semilla.click(fn=lambda: gr.update(choices=listar_archivos_semilla()), outputs=semilla_dropdown)

    def _cargar_semilla_ui(nombre):
        if not nombre:
            return ""
        datos = cargar_dataset_semilla(nombre)
        return json.dumps(datos, indent=2, ensure_ascii=False)

    btn_cargar_semilla.click(fn=_cargar_semilla_ui, inputs=semilla_dropdown, outputs=semilla_texto)

    def _formatear_semilla_ui(texto, esquema):
        datos = formatear_semilla_con_ia(texto, esquema)
        return json.dumps(datos, indent=2, ensure_ascii=False)

    btn_formatear_semilla.click(fn=_formatear_semilla_ui, inputs=[semilla_texto, esquema_activo], outputs=semilla_texto)

    def _guardar_semilla_y_usar(texto, nombre):
        datos = json.loads(texto)
        nombre_final = nombre.strip() if nombre and nombre.strip() else generar_nombre_archivo("semilla", "json")
        if not nombre_final.lower().endswith(".json"):
            nombre_final += ".json"
        guardar_dataset_semilla(datos, nombre_final)
        return nombre_final, f"✅ Semilla activa: {nombre_final}", nombre_final

    btn_guardar_semilla.click(
        fn=_guardar_semilla_y_usar,
        inputs=[semilla_texto, semilla_nombre_guardar],
        outputs=[semilla_activa, semilla_estado, semilla_activa_display]
    )

    # ══════════════ Eventos: pestaña Generar ══════════════
    def _generar_ui(semilla, esquema, prompt, num_ejemplos, modelo, formato_salida, instrucciones, nombre_salida):
        if not semilla or not esquema or not prompt:
            return "", "", "❌ Faltan ficheros activos: revisa las pestañas Prompt, Esquema y Semilla."
        try:
            registros, ruta, errores = generar_dataset_sintetico(
                num_ejemplos=int(num_ejemplos),
                modelo=modelo,
                formato_salida=formato_salida,
                nombre_semilla=semilla,
                esquema=esquema,
                system_prompt=prompt,
                nombre_salida=nombre_salida if nombre_salida.strip() != "" else None,
                instrucciones_adicionales=instrucciones
            )
            preview = json.dumps(registros[:5], indent=2, ensure_ascii=False)
            estado = f"✅ {len(registros)} registros generados, {len(errores)} con error.\nGuardado en:\n{ruta}"
            return preview, ruta, estado
        except Exception as e:
            return "", "", f"❌ Error: {str(e)}"

    btn_generar.click(
        fn=_generar_ui,
        inputs=[semilla_activa, esquema_activo, prompt_activo, num_ejemplos, modelo,
                formato_salida, instrucciones_adicionales, nombre_salida],
        outputs=[preview_output, ruta_output, estado_output]
    )


In [41]:
app.launch(share=True) #, debug=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://614801fd469e0b25b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Diagnóstico manual - pégalo en una celda nueva y ejecútalo
seed_dataset = cargar_dataset_semilla("semilla_20260703_164338.json")
esquema = cargar_esquema("esquema_20260703_164226.json")
prompt = cargar_prompt("prompt_20260703_163919.json")

siguiente_id = calcular_siguiente_id(seed_dataset, esquema.get("campo_id"))
texto_crudo = generar_con_claude(prompt, seed_dataset, 10, siguiente_id)

print("=== RESPUESTA CRUDA DE CLAUDE ===")
print(texto_crudo)

modelo_validacion = construir_modelo_validacion(esquema)
validos, errores = parsear_y_validar(texto_crudo, modelo_validacion)

print(f"Válidos: {len(validos)}")
print(f"Errores: {len(errores)}")
for e in errores:
    print("-", e)

print("\n=== Campos del esquema ===")
print([c["nombre"] for c in esquema["campos"]])
print("\n=== Campos del primer registro semilla ===")
print(list(seed_dataset[0].keys()))